In [1]:
import pandas as pd
from collections import Counter

# Load the dataset
def load_data(file_path):
    return pd.read_csv(file_path)

# Save the fixed dataset
def save_data(data, output_path):
    data.to_csv(output_path, index=False)

# Detect and fix mismatches
def fix_label_mismatches(data):
    # Define patterns
    sql_injection_pattern = "UNION|SELECT|INSERT|DELETE|UPDATE"
    xss_pattern = "<script>|%3Cscript%3E"
    dir_traversal_pattern = r"\..\/|\..\\"

    # Detection columns
    data['detected_sql_injection'] = data['request'].str.contains(sql_injection_pattern, case=False, na=False)
    data['detected_xss'] = data['request'].str.contains(xss_pattern, case=False, na=False)
    data['detected_dir_traversal'] = data['request'].str.contains(dir_traversal_pattern, case=False, na=False)

    # Brute Force Detection
    ip_attempts = Counter(data['client_ip'])
    brute_force_threshold = 5
    data['detected_brute_force'] = data['client_ip'].map(lambda ip: ip_attempts[ip] > brute_force_threshold)

    # DoS Detection
    dos_rate_threshold = 100
    data['request_rate'] = data.groupby('client_ip')['client_ip'].transform('count')
    data['detected_dos'] = data['request_rate'] > dos_rate_threshold

    # Revalidate labels
    def determine_revalidated_label(row):
        if row['detected_sql_injection']:
            return 'SQL Injection'
        if row['detected_xss']:
            return 'XSS'
        if row['detected_dir_traversal']:
            return 'Directory Traversal'
        if row['detected_brute_force']:
            return 'Brute Force'
        if row['detected_dos']:
            return 'DoS'
        return 'Normal'

    data['revalidated_label'] = data.apply(determine_revalidated_label, axis=1)

    # Fix mismatches
    data['corrected_label'] = data['revalidated_label']

    # Drop helper columns
    data = data.drop(columns=[
        'detected_sql_injection', 'detected_xss', 'detected_dir_traversal', 
        'detected_brute_force', 'detected_dos', 'request_rate', 'revalidated_label'
    ], errors='ignore')

    return data

# Main function (for Jupyter Notebook)
def process_data_in_notebook(input_file, output_file):
    # Load data
    data = load_data(input_file)

    # Fix label mismatches
    fixed_data = fix_label_mismatches(data)

    # Save the fixed dataset
    save_data(fixed_data, output_file)
    print(f"Fixed dataset saved to {output_file}")

# Example usage
process_data_in_notebook(input_file='./dataset/corrected_labeled_logs.csv', output_file='./dataset/fixed_labeled_logs.csv')


Fixed dataset saved to ./dataset/fixed_labeled_logs.csv


**LABELLING 5**


In [5]:
import pandas as pd
from collections import Counter

# Load the dataset
def load_data(file_path):
    return pd.read_csv(file_path)

# Save the labeled dataset
def save_data(data, output_path):
    data.to_csv(output_path, index=False)

# Labeling function to detect multiple attack types
def label_logs(data, brute_force_threshold=5, dos_rate_threshold=100, time_window=None):
    # Define patterns
    sql_injection_pattern = r"UNION|SELECT|INSERT|DELETE|UPDATE|DROP|CREATE|ALTER|--|/\*.*?\*/|%27"
    xss_pattern = r"<script>|%3Cscript%3E|javascript:|onload|eval|src=['\"]javascript:"
    dir_traversal_pattern = r"\..\/|\..\\|%2e%2e%2f|%2e%2e%5c|etc/passwd|boot.ini"

    # Add metadata for analysis
    data['num_params'] = data['request'].str.count('&') + 1
    data['url_length'] = data['request'].str.len()

    # Detection columns
    data['detected_sql_injection'] = data['request'].str.contains(sql_injection_pattern, case=False, na=False)
    data['detected_xss'] = data['request'].str.contains(xss_pattern, case=False, na=False)
    data['detected_dir_traversal'] = data['request'].str.contains(dir_traversal_pattern, case=False, na=False)

    # Brute Force Detection with time window (if provided)
    ip_attempts = Counter(data['client_ip'])
    data['detected_brute_force'] = data['client_ip'].map(lambda ip: ip_attempts[ip] > brute_force_threshold)

    if time_window:
        data['timestamp'] = pd.to_datetime(data['timestamp'], format="%d/%b/%Y:%H:%M:%S %z", errors='coerce')
        data.sort_values(by=['client_ip', 'timestamp'], inplace=True)
        data['time_diff'] = data.groupby('client_ip')['timestamp'].diff().dt.total_seconds()
        data['brute_force_time_window'] = data['time_diff'] <= time_window
        data['detected_brute_force'] = data['detected_brute_force'] & data['brute_force_time_window']

    # DoS Detection with rate threshold
    data['request_rate'] = data.groupby('client_ip')['client_ip'].transform('count')
    data['detected_dos'] = data['request_rate'] > dos_rate_threshold

    # Determine labels based on detection
    def determine_label(row):
        if row['detected_sql_injection']:
            return 'SQL Injection'
        if row['detected_xss']:
            return 'XSS'
        if row['detected_dir_traversal']:
            return 'Directory Traversal'
        if row['detected_brute_force']:
            return 'Brute Force'
        if row['detected_dos']:
            return 'DoS'
        return 'Normal'

    data['label'] = data.apply(determine_label, axis=1)

    # Drop helper columns
    data = data.drop(columns=[
        'detected_sql_injection', 'detected_xss', 'detected_dir_traversal', 
        'detected_brute_force', 'detected_dos', 'request_rate', 'brute_force_time_window', 'time_diff'
    ], errors='ignore')

    return data

# Main function for processing
def process_logs(input_file, output_file, time_window=None):
    # Load data
    data = load_data(input_file)

    # Label the logs with optional time window for brute force detection
    labeled_data = label_logs(data, time_window=time_window)

    # Save the labeled dataset
    save_data(labeled_data, output_file)
    print(f"Labeled dataset saved to {output_file}")

# Example usage
process_logs(input_file='./dataset/corrected_labeled_logs.csv', output_file='./dataset/logs_last.csv', time_window=60)

Labeled dataset saved to ./dataset/logs_last.csv


In [3]:
import pandas as pd

# Load the dataset
file_path = './dataset/logs_last.csv'  # Ganti dengan path file Anda
data = pd.read_csv(file_path)

# Drop the existing 'label' column if it exists and rename 'corrected_label' to 'label'
if 'label' in data.columns:
    data = data.drop(columns=['label'])
if 'corrected_label' in data.columns:
    data = data.rename(columns={'corrected_label': 'label'})

# Save the modified dataset to a new file
output_path = './dataset/logs_last_modified.csv'
data.to_csv(output_path, index=False)

# Display the first few rows of the modified dataset
print("Modified Dataset (Preview):")
print(data.head())


Modified Dataset (Preview):
    client_ip                  timestamp method  \
0   10.19.0.3  2023-04-14 10:10:47+07:00    GET   
1   10.19.0.3  2023-04-14 16:45:17+07:00    GET   
2  10.200.7.2  2023-04-12 07:49:00+07:00    GET   
3  10.200.7.2  2023-04-12 07:49:03+07:00    GET   
4  10.200.7.2  2023-04-12 07:59:01+07:00    GET   

                                             request  protocol  status  \
0  /yayasan-pendidikan-telkom-siap-hadirkan-insti...  HTTP/1.0     200   
1                                                  /  HTTP/1.0     200   
2                                                  /  HTTP/1.0     200   
3                                       /favicon.ico  HTTP/1.0     302   
4                                                  /  HTTP/1.0     200   

   response_size                     referrer  \
0         182649      https://www.google.com/   
1         258016                            -   
2         257487      https://www.google.com/   
3              0  https: